In this notebook, we will train a simple logistic regression model and dump it as a .joblib file for use in Matlab. 

# Training a simple logistic regression model


In [51]:
import os
import pandas as pd

# Define data reading function.
def combine_csv(folder_path, test_condition):
    """
    Combine all CSV files in a folder into a single DataFrame.
    :param folder_path: Path to the folder containing the CSV files
    :param seq_idx: Sequence index
    :param label: Label of the sequence (Normal - 0, Abnormal - 1)
    :return: A single DataFrame containing all the data from the CSV files
    """

    # Get a list of all CSV files in the folder
    csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

    # Create an empty DataFrame to store the combined data
    combined_df = pd.DataFrame()

    # Iterate over the CSV files in the folder
    for file in csv_files:
        # Construct the full path to each CSV file
        file_path = os.path.join(folder_path, file)

        # Read each CSV file into a DataFrame
        df = pd.read_csv(file_path)
        # Drop the time. Will add later.
        df = df.drop(labels=df.columns[0], axis=1)

        # Extract the file name (excluding the extension) to use as a prefix
        file_name = os.path.splitext(file)[0]

        # Add a prefix to each column based on the file name
        df = df.add_prefix(f'{file_name}_')

        # Concatenate the current DataFrame with the combined DataFrame
        combined_df = pd.concat([combined_df, df], axis=1)

    df = pd.read_csv(file_path)
    combined_df = pd.concat([df['time'], combined_df], axis=1)
    combined_df.loc[:, 'test_condition'] = test_condition

    return combined_df       

In this code section, we specify the training data. In this sample, we only train a fault dection model for motor 4. So we only use 'static_with_fault_4' and 'steady_state_after_movement' as trainig data, as the other dataset does not have motor 4 failures.

In [52]:
# Read the data
# path_training = ['static_with_fault_1', 'static_with_fault_2', 'static_with_fault_3', 
# 'static_with_fault_4', 'static_with_fault_5', 'static_with_fault_6', 
# 'steady_state_after_movement', 'steady_state_not_moving'
# ]

path_training = ['motor1_1', 'motor2_1','motor3_1','motor4_1','motor5_1','motor6_1',]
path_header = '../data_collection/collected_data/'

# Read data
df = pd.DataFrame()
for tmp_path in path_training:
    path = path_header + tmp_path
    tmp_df = combine_csv(path, tmp_path)
    df = pd.concat([df, tmp_df])
    df = df.reset_index(drop=True)

df['temperature_1s_ago'] = df['data_motor_5_temperature'].shift(10)
df['temperature_2s_ago'] = df['data_motor_5_temperature'].shift(20)
df['temperature_3s_ago'] = df['data_motor_5_temperature'].shift(30)

df = df.dropna()

We can use a PCA to visualize the patterns of the two classes. It can be seen that parts of the positive samples are mixed with negative ones.

In [53]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Separate the features (X) and the target variable (y)
X = df.drop(['test_condition', 'time', 'data_motor_1_label', 'data_motor_2_label', 'data_motor_3_label', 
             'data_motor_4_label', 'data_motor_5_label', 'data_motor_6_label'], axis=1).values
y = df['data_motor_5_label'].values


We train a simple logistic regression model with scikit-learn. A pipeline is used to preprocess the data: the data are first standardized, and used for training and predition. We use the balanced class weights to consider the problem of inbalanced labels. We use GridSearchCV for hyper-parameter tuning.

In [54]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create a pipeline with Standardization and Logistic Regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])

# Define hyperparameters for grid search
param_grid = {'classifier__C': np.logspace(-3, 2, 10)}

# Use GridSearchCV to find the best hyperparameters and fit the pipeline
grid_search = GridSearchCV(pipeline, param_grid, cv=4, scoring='f1')
grid_search.fit(X_train, y_train)

# Use grid_search.predict to make predictions on the testing dataset
y_pred = grid_search.predict(X_test)

# Compute confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Compute precision
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision}")

# Compute recall
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall}")

# Compute F1 score
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1}")


Confusion Matrix:
[[417   6]
 [  0  31]]
Accuracy: 0.986784140969163
Precision: 0.8378378378378378
Recall: 1.0
F1 Score: 0.911764705882353


Can you try to train on the training dataset, except for the "task_fault"? And you test on "task_fault" only?

In [58]:
path_testing = ['fault_5']
path_header = '../data_collection/collected_data/'

# Read data
df = pd.DataFrame()
for tmp_path in path_testing:
    path = path_header + tmp_path
    tmp_df = combine_csv(path, tmp_path)
    df = pd.concat([df, tmp_df])
    df = df.reset_index(drop=True)


df['temperature_1s_ago'] = df['data_motor_5_temperature'].shift(10)
df['temperature_2s_ago'] = df['data_motor_5_temperature'].shift(20)
df['temperature_3s_ago'] = df['data_motor_5_temperature'].shift(30)
df = df.dropna()
# Separate the features (X) and the target variable (y)
X_test = df.drop(['test_condition', 'time', 'data_motor_1_label', 'data_motor_2_label', 'data_motor_3_label', 
             'data_motor_4_label', 'data_motor_5_label', 'data_motor_6_label'], axis=1).values
y_test = df['data_motor_5_label'].values

# Use grid_search.predict to make predictions on the testing dataset
y_pred = grid_search.predict(X_test)

# Compute confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Compute precision
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision}")

# Compute recall
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall}")

# Compute F1 score
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1}")

Confusion Matrix:
[[ 30   0]
 [466   0]]
Accuracy: 0.06048387096774194
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


c:\Users\aa997\.conda\envs\robotic\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# Dump the model as a joblib file.

Now let's output the model to a file. Later, this file will be used in the Matlab GUI for real-time prediction.

In [56]:
import joblib

# We save the model as 'clf_mdl.joblib':
file_name = '../fault_detection_evaluation/detection_mdl_motor_4.joblib'
joblib.dump(grid_search, file_name)

['../fault_detection_evaluation/detection_mdl_motor_4.joblib']

We can load the model and see if it works. Note that the classification is made using clf, which is loaded from your device.

In [57]:
import joblib

file_name = '../fault_detection_evaluation/detection_mdl_motor_4.joblib'

clf = joblib.load(file_name)
y_pred = clf.predict(X_test)

# Compute confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Compute precision
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision}")

# Compute recall
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall}")

# Compute F1 score
f1 = f1_score(y_test, y_pred)
print(f"F1 Score: {f1}")

Confusion Matrix:
[[370]]
Accuracy: 1.0
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


c:\Users\aa997\.conda\envs\robotic\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\aa997\.conda\envs\robotic\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\aa997\.conda\envs\robotic\lib\site-packages\sklearn\metrics\_classification.py:1760: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
